# Unit 1 Practical: A Vehicle Management Class Hierarchy

Ian Stevens, Advanced Object-Oriented Design and Programming, Unit 1.

The seminar task was to implement a basic class hierarchy in Python for a simple vehicle management system, with `Vehicle` as the base class and `Car`, `Bike` and `Truck` as subclasses, and to use it to show inheritance, polymorphism, abstraction and encapsulation, together with constructors, destructors and access control.

The classes are in [`vehicles.py`](vehicles.py) and the tests in [`test_vehicles.py`](test_vehicles.py). This notebook walks through what each part shows. I wrote down simple sentences about the system first and took classes from the nouns and methods from the verbs, as Lott (2024, ch. 7) suggests:

* A fleet has vehicles, and each vehicle has a registration, a make, a model, a year and a VIN.
* A vehicle records trips on its odometer, which only goes forwards.
* A car carries passengers; a bike is a motorbike; a truck carries a load up to its payload limit.
* A driver may only be assigned to a vehicle that their licence covers.

`Bike` is taken to mean a motorbike, because a fleet register is about registered road vehicles. A bicycle has no registration, VIN or licence category, so it would not fit the base class at all. That is a point about inheritance in itself: the base class encodes assumptions that every subclass has to honour.

In [1]:
import gc
import platform

from vehicles import Bike, Car, Driver, Fleet, Truck, Vehicle

print("Python", platform.python_version(), platform.python_implementation())

Python 3.13.16 CPython


## 1. Constructors

Each subclass constructor checks its own arguments, then calls `super().__init__()` so that the base class validates and stores what every vehicle shares. Arguments after `*` must be given by keyword, which makes calls like `Truck(..., gross_weight_kg=18000, kerb_weight_kg=8000)` hard to get the wrong way round.

In [2]:
car = Car("ab12 cde", "Ford", "Focus", 2021, vin="WF0XXXGCDX1234567", passenger_seats=4)
bike = Bike("BK19ABC", "Honda", "CB125R", 2022, vin="JH2JC7910NK000123", engine_cc=125, power_kw=11)
truck = Truck("TR70XYZ", "DAF", "LF", 2020, vin="XLRAE45CE0L123456",
              gross_weight_kg=18000, kerb_weight_kg=8000)

for vehicle in (car, bike, truck):
    print(repr(vehicle))

Car(registration='AB12CDE', make='Ford', model='Focus', year=2021)
Bike(registration='BK19ABC', make='Honda', model='CB125R', year=2022)
Truck(registration='TR70XYZ', make='DAF', model='LF', year=2020)


Invalid objects are refused at construction, so a `Vehicle` that exists is always a valid one.

In [3]:
attempts = {
    "bad registration": lambda: Car("NOT A REG!", "Ford", "Focus", 2021, vin="WF0XXXGCDX1234567"),
    "nine passenger seats": lambda: Car("AB12CDE", "Ford", "Transit", 2021, vin="WF0XXXGCDX1234567", passenger_seats=9),
    "3,500 kg truck": lambda: Truck("TR70XYZ", "Ford", "Transit", 2020, vin="WF0XXXGCDX1234567",
                                    gross_weight_kg=3500, kerb_weight_kg=2000),
    "VIN containing O": lambda: Car("AB12CDE", "Ford", "Focus", 2021, vin="WF0OOOGCDX1234567"),
}
for label, attempt in attempts.items():
    try:
        attempt()
    except (ValueError, TypeError) as error:
        print(f"{label:22} -> {type(error).__name__}: {error}")

bad registration       -> ValueError: invalid registration: 'NOT A REG!'
nine passenger seats   -> ValueError: a car has 1 to 8 passenger seats; more than that is a minibus and needs a D1 or D licence
3,500 kg truck         -> ValueError: a truck must weigh more than 3,500 kg fully loaded; lighter goods vehicles are driven on a category B licence
VIN containing O       -> ValueError: invalid VIN: expected 17 characters, no I, O or Q


## 2. Abstraction

`Vehicle` is an abstract base class with two abstract properties. Python will not create a `Vehicle`, or a subclass that leaves either property out.

In [4]:
print("Abstract members:", sorted(Vehicle.__abstractmethods__))
try:
    Vehicle("AB12CDE", "Ford", "Focus", 2021, vin="WF0XXXGCDX1234567")
except TypeError as error:
    print("TypeError:", error)

Abstract members: ['licence_category', 'wheels']
TypeError: Can't instantiate abstract class Vehicle without an implementation for abstract methods 'licence_category', 'wheels'


## 3. Inheritance

`Car`, `Bike` and `Truck` inherit the constructor checks, the odometer, `describe()` and `__repr__()` from `Vehicle`. The method resolution order shows where Python looks for a method.

In [5]:
for cls in (Car, Bike, Truck):
    print(cls.__name__, "->", " -> ".join(c.__name__ for c in cls.__mro__[1:]))

print()
print("Inherited, not redefined:", [name for name in ("record_trip", "describe", "__repr__")
                                    if name not in Car.__dict__])

Car -> Vehicle -> ABC -> object
Bike -> Vehicle -> ABC -> object
Truck -> Vehicle -> ABC -> object

Inherited, not redefined: ['record_trip', 'describe', '__repr__']


## 4. Polymorphism

`Fleet.report()` calls `describe()` on every vehicle without checking its type. `describe()` is a template method in the base class that asks each subclass for `wheels`, `licence_category` and the protected hook `_extra_details()`.

In [6]:
fleet = Fleet("North depot")
for vehicle in (car, bike, truck):
    fleet.add(vehicle)

car.record_trip(42.5)
truck.load(6_500)
truck.record_trip(310)

for line in fleet.report():
    print(line)

2021 Ford Focus (AB12CDE): 4 wheels, licence B, 4 passenger seats, 42.5 km
2022 Honda CB125R (BK19ABC): 2 wheels, licence A1, 125 cc, 11 kW, 0.0 km
2020 DAF LF (TR70XYZ): 6 wheels, licence C, 6,500 of 10,000 kg loaded, 310.0 km


The licence category is the clearest case of each class answering the same question in its own way. For bikes it depends on engine size and power, so the boundaries are worth checking (rules simplified from the DVLA categories).

In [7]:
for cc, kw in [(125, 11), (125, 11.1), (650, 35), (650, 35.1), (0, 10)]:
    b = Bike("BK19ABC", "Test", "Bike", 2022, vin="JH2JC7910NK000123", engine_cc=cc, power_kw=kw)
    print(f"{cc:>4} cc {kw:>5} kW -> {b.licence_category}")

 125 cc    11 kW -> A1
 125 cc  11.1 kW -> A2
 650 cc    35 kW -> A2
 650 cc  35.1 kW -> A
   0 cc    10 kW -> A1


## 5. Encapsulation

The odometer has a getter but no setter, and `record_trip()` only accepts positive, finite distances. Compare Matthes's (2023, ch. 9) `Car`, where `update_odometer()` refuses to roll the reading back but `odometer_reading` is a public attribute that any caller can overwrite.

In [8]:
class MatthesCar:
    # Simplified from Matthes (2023, ch. 9)
    def __init__(self):
        self.odometer_reading = 0

    def update_odometer(self, mileage):
        if mileage >= self.odometer_reading:
            self.odometer_reading = mileage
        else:
            print("You can't roll back an odometer!")

m = MatthesCar()
m.update_odometer(23_500)
m.update_odometer(100)        # refused by the method...
m.odometer_reading = 100      # ...but nothing stops direct assignment
print("Matthes-style car:", m.odometer_reading)

for attempt in ("car.odometer_km = 0", "car.record_trip(-40)", "car.record_trip(float('nan'))"):
    try:
        exec(attempt)
    except (AttributeError, ValueError) as error:
        print(f"{attempt:32} -> {type(error).__name__}: {error}")
print("My car:", car.odometer_km, "km")

You can't roll back an odometer!
Matthes-style car: 100
car.odometer_km = 0              -> AttributeError: property 'odometer_km' of 'Car' object has no setter
car.record_trip(-40)             -> ValueError: distance_km must be a positive, finite number
car.record_trip(float('nan'))    -> ValueError: distance_km must be a positive, finite number
My car: 42.5 km


The truck's load is protected in the same way, and a refused load leaves the state unchanged.

In [9]:
try:
    truck.load(4_000)
except ValueError as error:
    print("ValueError:", error)
print("Still loaded:", truck.load_kg, "kg of", truck.payload_capacity_kg)

ValueError: overload: 10,500 kg exceeds the 10,000 kg payload
Still loaded: 6500.0 kg of 10000


## 6. Access control

Python has no `private` or `protected` keywords. Lott and Phillips (2021, ch. 2) describe the leading underscore as a convention and name mangling as a strong hint rather than a guarantee. This cell shows what each level actually does.

In [10]:
print("public    car.make               ->", car.make)
print("protected car._odometer_km       ->", car._odometer_km, "(readable; the underscore is only a warning)")
print("private   hasattr(car, '__vin')  ->", hasattr(car, "__vin"))
print("private   car._Vehicle__vin      ->", car._Vehicle__vin, "(name mangling, not protection)")
print()
print("vin_masked ->", car.vin_masked)
print("repr       ->", repr(car))
print("matches_vin('wf0xxxgcdx1234567') ->", car.matches_vin("wf0xxxgcdx1234567"))

public    car.make               -> Ford
protected car._odometer_km       -> 42.5 (readable; the underscore is only a warning)
private   hasattr(car, '__vin')  -> False
private   car._Vehicle__vin      -> WF0XXXGCDX1234567 (name mangling, not protection)

vin_masked -> *************4567
repr       -> Car(registration='AB12CDE', make='Ford', model='Focus', year=2021)
matches_vin('wf0xxxgcdx1234567') -> True


So the single underscore did not stop the odometer being wound back by a caller who ignores the convention:

In [11]:
car._odometer_km = 0.0
print(car.odometer_km)
car.record_trip(42.5)  # put it back

0.0


In Python, then, access control is about making the right use easy and the wrong use visible, not about stopping a determined caller. Real protection for something like an odometer would need a boundary outside the object. The more useful security work in the class is elsewhere: keeping the VIN out of `__repr__()` and error messages so it does not leak into logs, and refusing invalid data at the constructor.

Access control also applies to the domain. `Fleet.assign()` checks every assignment against the driver's licence, and an unknown category entitles the driver to nothing.

In [12]:
sam = Driver("Sam", frozenset({"B"}))
alex = Driver("Alex", frozenset({"B", "C"}))

for registration, driver in [("AB12CDE", sam), ("TR70XYZ", sam), ("TR70XYZ", alex), ("BK19ABC", alex)]:
    try:
        fleet.assign(registration, driver)
        print(f"{driver.name} -> {registration}: assigned")
    except PermissionError as error:
        print(f"{driver.name} -> {registration}: refused ({error})")

Sam -> AB12CDE: assigned
Sam -> TR70XYZ: refused (Sam is not licensed for category C (TR70XYZ))
Alex -> TR70XYZ: assigned
Alex -> BK19ABC: refused (Alex is not licensed for category A1 (BK19ABC))


## 7. Destructors

Python's `__del__()` is a finaliser. The language reference calls it '(improperly) a destructor' and notes that `del x` only removes one reference; the method runs when the last reference goes (Python Software Foundation, 2026). `Vehicle.__del__()` records the registration so this can be seen. These results rely on CPython's reference counting.

In [13]:
Vehicle.decommissioned.clear()

spare = Car("SP11ARE", "Kia", "Picanto", 2019, vin="KNAB2511AKT123456")
del spare
print("after del of the only reference:   ", Vehicle.decommissioned)

held = Car("HE11DDD", "Kia", "Ceed", 2020, vin="KNAB2511AKT654321")
fleet.add(held)
del held
print("after del while the fleet holds it:", Vehicle.decommissioned)

fleet.remove("HE11DDD")
print("after the fleet lets go:           ", Vehicle.decommissioned)

after del of the only reference:    ['SP11ARE']
after del while the fleet holds it: ['SP11ARE']
after the fleet lets go:            ['SP11ARE', 'HE11DDD']


Testing the finaliser exposed a case I had not expected. Python runs `__del__()` on an object whose constructor raised, because the object already exists by the time `__init__()` runs. If a subclass calls `super().__init__()` first and validates afterwards, a car that was never valid is recorded as decommissioned:

In [14]:
class EagerCar(Car):
    # Calls the base constructor first, then validates its own argument.
    def __init__(self, *args, colour, **kwargs):
        super().__init__(*args, **kwargs)
        if colour not in {"red", "blue"}:
            raise ValueError("unknown colour")

Vehicle.decommissioned.clear()
try:
    EagerCar("BA11DDD", "Kia", "Ceed", 2020, vin="KNAB2511AKT654321", colour="green")
except ValueError:
    pass
gc.collect()
print("half-built object finalised:", Vehicle.decommissioned)

half-built object finalised: ['BA11DDD']


`Car`, `Bike` and `Truck` avoid this by validating their own arguments before calling `super().__init__()`, and `Vehicle` sets `_registration` last. The wider lesson is not to put anything that matters in `__del__()`: cleanup that must happen belongs in an explicit method or a context manager (Lott, 2024, ch. 7).

## 8. Tests

The tests cover each concept above, and the doctest in the module docstring runs with them, as in Lott (2024, ch. 15).

In [15]:
import subprocess, sys
result = subprocess.run([sys.executable, "-m", "unittest", "test_vehicles"],
                        capture_output=True, text=True)
print(result.stderr.strip().splitlines()[-3:])

['Ran 28 tests in 0.014s', '', 'OK']


## References

Lott, S.F. (2024) *Modern Python cookbook*. 3rd edn. Birmingham: Packt.

Lott, S.F. and Phillips, D. (2021) *Python object-oriented programming*. 4th edn. Birmingham: Packt.

Matthes, E. (2023) *Python crash course*. 3rd edn. San Francisco, CA: No Starch Press.

Python Software Foundation (2026) *The Python language reference: 3. Data model*. Available at: https://docs.python.org/3/reference/datamodel.html (Accessed: 20 October 2026).